# Basic steps to visualize metabolic fluxes

##Set up Google Colab

In [1]:
!pip install cobra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.8 MB/s eta 0:00:00


In [2]:
import cobra
from cobra.io import load_json_model
import json
from google.colab import files

In [3]:
#2. Download the iJO1366 model
!wget -nc http://bigg.ucsd.edu/static/models/iJO1366.json

--2026-06-16 04:51:35--  http://bigg.ucsd.edu/static/models/iJO1366.json
Resolving bigg.ucsd.edu (bigg.ucsd.edu)... 169.228.33.117
Connecting to bigg.ucsd.edu (bigg.ucsd.edu)|169.228.33.117|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2948407 (2.8M) [application/json]
Saving to: ‘iJO1366.json’

iJO1366.json        100%[===================>]   2.81M  6.45MB/s    in 0.4s    

2026-06-16 04:51:36 (6.45 MB/s) - ‘iJO1366.json’ saved [2948407/2948407]



In [4]:
#Load model
model = load_json_model("iJO1366.json")

In [5]:
print("Number of reactions:", len(model.reactions))
print("Number of metabolites:", len(model.metabolites))
print("Current objective:", model.objective)

Number of reactions: 2583
Number of metabolites: 1805
Current objective: Maximize
1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


In [6]:
#3. Find reaction IDs
for reaction in model.reactions:
    if "glc" in reaction.id.lower():
        print(reaction.id, reaction.name)

EX_glc__D_e D-Glucose exchange
EX_glcn_e D-Gluconate exchange
EX_glcr_e D-Glucarate exchange
EX_glcur_e D-Glucuronate exchange
EX_glcur1p_e D-Glucuronate 1-phosphate exchange
EX_2ddglcn_e 2-Dehydro-3-deoxy-D-gluconate exchange
EX_5dglcn_e 5-Dehydro-D-gluconate exchange
EX_udpglcur_e UDP-D-glucuronate exchange
5DGLCNR 5-dehydro-D-gluconate reductase
5DGLCNt2rpp 5-Dehydro-D-gluconate transport via proton symport, reversible (periplasm)
5DGLCNtex 5-Dehydro-D-gluconate transport via diffusion (extracellular to periplasm)
DDGLCNt2rpp 2-dehydro-3-deoxy-D-gluconate transport via proton symport, reversible (periplasm)
DDGLCNtex 2-dehydro-3-deoxy-D-gluconate transport via diffusion (extracellular to periplasm)
DKGLCNR1 2,5-diketo-D-gluconate reductase
DKGLCNR2x 2,5-diketo-D-gluconate reductase (NADH)
DKGLCNR2y 2,5-diketo-D-gluconate reductase (NADPH)
GLCATr D-glucose O-acetyltransferase
GLCDpp Glucose dehydrogenase (ubiquinone-8 as acceptor) (periplasm)
GLCNt2rpp D-gluconate transport via proto

In [16]:
#Search any reaction by matching a word
for reaction in model.reactions:
    if "biomass" in reaction.id.lower():
        print(reaction.id, reaction.name)

BIOMASS_Ec_iJO1366_WT_53p95M E. coli biomass objective function (iJO1366) - WT - with 53.95 GAM estimate
BIOMASS_Ec_iJO1366_core_53p95M E. coli biomass objective function (iJO1366) - core - with 53.95 GAM estimate


In [8]:
#4. Modify lower and upper bounds
eaction = model.reactions.get_by_id("EX_glc__D_e")

reaction.lower_bound = -10
reaction.upper_bound = 1000

In [9]:
print(reaction.id)
print("Lower bound:", reaction.lower_bound)
print("Upper bound:", reaction.upper_bound)

EX_glc__D_e
Lower bound: -10
Upper bound: 1000


In [10]:
#5. Run optimization
solution = model.optimize()

In [11]:
print("Status:", solution.status)
print("Objective value:", solution.objective_value)

Status: optimal
Objective value: 0.9823718127269785


In [12]:
solution.fluxes

,fluxes
EX_cm_e,0.000000
EX_cmp_e,0.000000
EX_co2_e,19.675223
EX_cobalt2_e,-0.000025
DM_4crsol_c,0.000219
...,...
RNDR4,0.000000
RNDR4b,0.000000
RNTR1c2,0.025705
RNTR2c2,0.026541


In [14]:
#6. Save fluxes as JSON for Escher
flux_dictionary = solution.fluxes.to_dict()

In [15]:

with open("iJO1366_fluxes.json", "w") as f:
    json.dump(flux_dictionary, f)

In [ ]:
#files.download("iJO1366_fluxes.json")